# Ablação Tier 2 — N=5000, params FIXOS, 30 seeds — **só Transformers (GPU)**

Parte GPU da ablação Tier 2 em **N=5000** (params fixos = moda do GridCV N=2000).
Roda **apenas os 6 Transformers** — os LSSVMs + XGBoost são rodados à parte na CPU
e os dois JSONs são mesclados na análise.

**Transformers:** FT-Softmax, FT-TopK, FT-Entmax, FT-Sparsemax, SAINT, FT-CUR  
**Datasets:** todos os 6 (TELCO ~4922 treino; os outros 5 em 5000).  
**Saída:** `results/tier2_fixedparams_n5000_transformers.json` (resumível — pula o que já terminou).

**Custo estimado:** ~6–10 h GPU. Se a sessão de 12 h não bastar, baixe o JSON,
suba como dataset e re-execute (retoma de onde parou).

**Antes de rodar:** Settings → Accelerator → GPU T4 x2 (ou P100).

In [ ]:
# ── Célula 1: Verifica GPU ───────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Célula 2: Clonar repo ──────────────────────────────────────────
import os, subprocess
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', GIT_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -3
print('Dir:', os.getcwd())

In [ ]:
# ── Célula 3: Dependências ─────────────────────────────────────────
!pip install -q entmax einops xgboost
import torch, sklearn, numpy, xgboost
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | xgboost {xgboost.__version__}')

In [ ]:
# ── Célula 4: Gera o config de params fixos (moda do GridCV N=2000) ──────────
# Deriva de results/tier2_gridcv.json + tier2_transformers*.json (já commitados).
!python scripts/extract_tier2_fixed_params.py

In [ ]:
# ── Célula 5: Rodar SÓ os Transformers (6 variantes, 6 datasets, 30 seeds, N=5000) ─
# Resumível: re-execute esta célula se a sessão cair (pula o que já terminou).
!python -u scripts/run_tier2_fixedparams.py \
    --n-train 5000 \
    --models FTTransformer_softmax FTTransformer_topk FTTransformer_entmax \
             FTTransformer_sparsemax SAINTColnorm FTTransformerCURColnorm \
    --output results/tier2_fixedparams_n5000_transformers.json \
    2>&1 | tee -a /kaggle/working/tier2_fixedparams_tf.log

import shutil
shutil.copy('results/tier2_fixedparams_n5000_transformers.json',
            '/kaggle/working/tier2_fixedparams_n5000_transformers.json')
print('\nSalvo em /kaggle/working/tier2_fixedparams_n5000_transformers.json')

In [ ]:
# ── Célula 6: Resumo (F1-macro médio por Transformer, 30 seeds) ────────────
import json, collections, statistics
from pathlib import Path

recs = json.loads(Path('results/tier2_fixedparams_n5000_transformers.json').read_text())
ok = [r for r in recs if r.get('status') == 'ok']
by = collections.defaultdict(list)
for r in ok:
    by[r['variant']].append(r['test_f1_macro'])

print(f'Registros OK: {len(ok)} | modelos: {len(by)}')
print(f"\n{'Modelo':<26} {'F1-macro (média±dp)':>22} {'n':>4}")
print('-' * 56)
for v, fs in sorted(by.items(), key=lambda kv: -statistics.mean(kv[1])):
    m = statistics.mean(fs)
    s = statistics.pstdev(fs)
    print(f'{v:<26} {m:>13.4f} ± {s:.4f} {len(fs):>4}')